Tähän lätkäisen vain esimerkki koodia otsikoiden kera miten featureita voisi mahdollisesti käsitellä phase 3:ssa.

Tässä oletuksena, että data on jo ajettu df nimiseen DataFrameen

## duplikaatit pois

In [ ]:
df = df.drop_duplicates()

## Mileage käsittely

In [ ]:
#pudotetaan äärimmäisen suuret mittarilukemat
df = df[df['mileage'] <= 600000]


# Lisätään mileage_missing feature jotai voidaan halutessa käyttää
# korvataan puuttuvat mittarilukemat autojen vuosimallin mediaani mittarilukemalla
df['mileage_missing'] = df['mileage'].isnull().astype(int)
median_by_year = df.groupby('year')['mileage'].median()
df['mileage'] = df['mileage'].fillna(df['year'].map(median_by_year))

## Fuel type imputointi:

In [ ]:
# Lasketaan merkki-malli-vuosimalli yleisin fuel_type
mode_fuel = df.groupby(['make_name', 'model_name', 'year'])['fuel_type'] \
              .agg(lambda x: x.mode()[0] if not x.mode().empty else None)

fuel_fill = df.set_index(['make_name', 'model_name', 'year']).index.to_series().map(mode_fuel).reset_index(drop=True)

# Imputoi puuttuvat
df['fuel_type'] = df['fuel_type'].fillna(fuel_fill)

# jos ei löytynyt aikasemmalla, nyt pelkkä merkki-malli
mode_fuel_mm = df.groupby(['make_name', 'model_name'])['fuel_type'] \
    .agg(lambda x: x.mode()[0] if not x.mode().empty else None)

df['fuel_type'] = df['fuel_type'].fillna(
    df.set_index(['make_name', 'model_name']).index.to_series().map(mode_fuel_mm).reset_index(drop=True)
)

# pelkän merkin perusteella
mode_fuel_make = df.groupby('make_name')['fuel_type'] \
    .agg(lambda x: x.mode()[0] if not x.mode().empty else None)

df['fuel_type'] = df['fuel_type'].fillna(df['make_name'].map(mode_fuel_make))


# poistetaan loput 0.01% joille ei löydy merkinn perusteella:

df = df.dropna(subset=['fuel_type'])

## Autojen ikä feature ja km_per_year

In [ ]:
data_year = 2021
df['car_age'] = data_year - df['year']
df['km_per_year'] = df['mileage'] / df['car_age']

# edge case
df.loc[df['car_age'] == 0, 'km_per_year'] = df['mileage']

## Options count
Luulen, että pelkkä options_count toimii aika hyvin. (Ei erikseen safety, comfort jne)

In [ ]:
df['options_count'] = df['major_options'].fillna('').apply(lambda x: len([item for item in str(x).split(',') if item.strip() != '']))

## Bin-features
Näiden kanssa ChatGPT:n mukaan toimii erityisen hyvin tree-malleissa

In [ ]:
df['age_bin'] = pd.cut(df['car_age'], bins=[0,3,5,8,100], labels=[0,1,2,3])
df['mileage_bin'] = pd.cut(df['mileage'], bins=[0,20000,50000,100000,200000,1000000], labels=[0,1,2,3,4])

## Moottorin suorituskyky
Hevosvoimat suhteessa moottorinkokoon


In [ ]:

df['hp_per_displacement'] = df['horsepower'] / df['engine_displacement']

## Simplifioitu vaihteisto? Ei erikseen esim 7 speed automatic ja 9 speed, vaan kaikki automaatit automaatiksi jne.

In [ ]:
def simplify_transmission(x):
    if pd.isna(x):
        return 'Unknown'
    x = x.lower()
    
    if 'auto' in x:
        return 'Automatic'
    elif 'manual' in x:
        return 'Manual'
    elif 'cvt' in x:
        return 'CVT'
    else:
        return 'Other'

df['transmission_simple'] = df['transmission_display'].apply(simplify_transmission)

test.